In [1]:
import pandas as pd 
import numpy as np 
import os

In [25]:
RAW_DIR = 'data/raw/'
CLEAN_DIR = 'data/clean/'
os.makedirs(CLEAN_DIR, exist_ok=True)

In [7]:
matches_raw = pd.read_csv(RAW_DIR + '/results.csv')
print("Shape: ", matches_raw.shape)
print("\nColums: ", matches_raw.columns.tolist())
print("\nTypes: ", matches_raw.dtypes)
print("\nNulls count: ", matches_raw.isnull().sum())
print("\nSample rows")
matches_raw.head()

Shape:  (49287, 9)

Colums:  ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Types:  date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object

Nulls count:  date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

Sample rows


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [9]:
matches = matches_raw.copy()

# 1: Parse date
matches['date'] = pd.to_datetime(matches['date'])

# 2. Filter 2000–2025
matches = matches[
    (matches['date'] >= '2000') &
    (matches['date'] <= '2025')
].copy()

# 3. Drop rows with missing scores
matches = matches.dropna(subset=['home_score', 'away_score'])

# 4. Cast scores to integers
matches['home_score'] = matches['home_score'].astype(int)
matches['away_score'] = matches['away_score'].astype(int)

# 5. Derive outcome column
def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'
    
matches["outcome"] = matches.apply(get_outcome, axis=1)

# ── 6. Standardize team names (strip whitespace, title-case) ───────────────
matches["home_team"] = matches["home_team"].str.strip().str.title()
matches["away_team"] = matches["away_team"].str.strip().str.title()

# ── 7. Reset index ─────────────────────────────────────────────────────────
matches = matches.reset_index(drop=True)

In [12]:
print("=== CLEAN MATCH DATA ===")
print("Shape            :", matches.shape)
print("Date range       :", matches["date"].min(), "→", matches["date"].max())
print("Unique home teams:", matches["home_team"].nunique())
print("Unique away teams:", matches["away_team"].nunique())
print("\nOutcome distribution:")
print(matches["outcome"].value_counts())
print("\nNull counts:\n", matches.isnull().sum())
print("\nSample rows:")
print(matches.head(5).to_string())

=== CLEAN MATCH DATA ===
Shape            : (23995, 10)
Date range       : 2000-01-04 00:00:00 → 2024-12-31 00:00:00
Unique home teams: 311
Unique away teams: 302

Outcome distribution:
outcome
home_win    11544
away_win     6848
draw         5603
Name: count, dtype: int64

Null counts:
 date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
outcome       0
dtype: int64

Sample rows:
        date            home_team away_team  home_score  away_score tournament           city              country  neutral   outcome
0 2000-01-04                Egypt      Togo           2           1   Friendly          Aswan                Egypt    False  home_win
1 2000-01-07              Tunisia      Togo           7           0   Friendly          Tunis              Tunisia    False  home_win
2 2000-01-08  Trinidad And Tobago    Canada           0           0   Friendly  Port of Spain  Trinidad and Tobago    Fals

In [19]:
matches.to_csv(CLEAN_DIR + "/matches_clean.csv", index=False)
print("✅ Saved → data/clean/matches_clean.csv")

✅ Saved → data/clean/matches_clean.csv


In [20]:
ranking_raw = pd.read_csv(RAW_DIR + "/fifa_ranking.csv")

print("Shape:", ranking_raw.shape)
print("\nColumns:", ranking_raw.columns.tolist())
print("\nData types:\n", ranking_raw.dtypes)
print("\nNull counts:\n", ranking_raw.isnull().sum())
print("\nSample rows:")
ranking_raw.head(5)

Shape: (67261, 8)

Columns: ['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']

Data types:
 rank               float64
country_full        object
country_abrv        object
total_points       float64
previous_points    float64
rank_change          int64
confederation       object
rank_date           object
dtype: object

Null counts:
 rank               8
country_full       0
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64

Sample rows:


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,83.0,Guatemala,GUA,15.0,0.0,83,CONCACAF,1992-12-31
1,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
2,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
3,34.0,Austria,AUT,38.0,0.0,34,UEFA,1992-12-31
4,35.0,Colombia,COL,36.0,0.0,35,CONMEBOL,1992-12-31


In [21]:
ranking = ranking_raw.copy()

# 1. Parse date — try both common formats in this dataset
ranking["rank_date"] = pd.to_datetime(ranking["rank_date"], errors="coerce")

# 2. Drop any rows where date couldn't be parsed
ranking = ranking.dropna(subset=["rank_date"])

# 3. Filter to 2000–2024 to match our match data range
ranking = ranking[
    (ranking["rank_date"].dt.year >= 2000) &
    (ranking["rank_date"].dt.year <= 2024)
].copy()

# 4. Keep only the columns we need
#    The dataset has: rank, country_full, country_abrv, total_points,
#                     previous_points, rank_change, confederation
ranking = ranking[["rank_date", "rank", "country_full", "total_points", "confederation"]]

# 5. Rename for clarity and consistency with matches_clean
ranking = ranking.rename(columns={
    "country_full"  : "team",
    "total_points"  : "fifa_points",
    "rank"          : "fifa_rank"
})

# 6. Standardize team names the same way we did for matches
ranking["team"] = ranking["team"].str.strip().str.title()

# 7. Sort by team and date
ranking = ranking.sort_values(["team", "rank_date"]).reset_index(drop=True)

print("✅ Cleaning done")

✅ Cleaning done


In [22]:
print("=== CLEAN FIFA RANKING DATA ===")
print("Shape             :", ranking.shape)
print("Date range        :", ranking["rank_date"].min(), "→", ranking["rank_date"].max())
print("Unique teams      :", ranking["team"].nunique())
print("Unique confederations:", ranking["confederation"].nunique())
print("\nConfederation breakdown:")
print(ranking["confederation"].value_counts())
print("\nNull counts:\n", ranking.isnull().sum())
print("\nSample rows:")
ranking.head(10)

=== CLEAN FIFA RANKING DATA ===
Shape             : (54691, 5)
Date range        : 2000-01-19 00:00:00 → 2024-04-04 00:00:00
Unique teams      : 214
Unique confederations: 6

Confederation breakdown:
confederation
UEFA        14043
CAF         14006
AFC         12022
CONCACAF     9155
OFC          2825
CONMEBOL     2640
Name: count, dtype: int64

Null counts:
 rank_date        0
fifa_rank        8
team             0
fifa_points      0
confederation    0
dtype: int64

Sample rows:


,rank_date,fifa_rank,team,fifa_points,confederation
0,2003-01-15,204.0,Afghanistan,7.0,AFC
1,2003-02-19,203.0,Afghanistan,9.0,AFC
2,2003-03-26,198.0,Afghanistan,48.0,AFC
3,2003-04-23,198.0,Afghanistan,48.0,AFC
4,2003-05-21,199.0,Afghanistan,48.0,AFC
5,2003-06-25,199.0,Afghanistan,48.0,AFC
6,2003-07-30,199.0,Afghanistan,48.0,AFC
7,2003-08-27,198.0,Afghanistan,48.0,AFC
8,2003-09-24,198.0,Afghanistan,48.0,AFC
9,2003-10-22,198.0,Afghanistan,48.0,AFC


In [28]:
# See which rows have null fifa_rank
print("Rows with null fifa_rank:")
print(ranking[ranking["fifa_rank"].isnull()])

Rows with null fifa_rank:
       rank_date  fifa_rank            team  fifa_points confederation
1014  2023-10-26        NaN  American Samoa       900.27           OFC
16879 2023-10-26        NaN         Eritrea       855.56           CAF
16880 2023-11-30        NaN         Eritrea       855.56           CAF
16881 2023-12-21        NaN         Eritrea       855.56           CAF
16882 2024-02-15        NaN         Eritrea       855.56           CAF
16883 2024-04-04        NaN         Eritrea       855.56           CAF
41206 2023-10-26        NaN           Samoa       894.26           OFC
49636 2023-10-26        NaN           Tonga       861.81           OFC


In [29]:
# Only 8 rows out of 54,691 — safe to drop them
ranking = ranking.dropna(subset=["fifa_rank"])

# Cast fifa_rank to integer now that nulls are gone
ranking["fifa_rank"] = ranking["fifa_rank"].astype(int)

print(f"Rows after dropping null ranks: {len(ranking):,}")
print(f"Null counts:\n{ranking.isnull().sum()}")
print("✅ No more nulls")

Rows after dropping null ranks: 54,683
Null counts:
rank_date        0
fifa_rank        0
team             0
fifa_points      0
confederation    0
dtype: int64
✅ No more nulls


In [31]:
ranking.to_csv(CLEAN_DIR + "ranking_clean.csv", index=False)
print("✅ Saved → data/clean/ranking_clean.csv")

✅ Saved → data/clean/ranking_clean.csv


In [32]:
# 48 qualified teams across 16 groups (A–P), 3 teams per group
# Source: FIFA official 2026 qualification results

wc2026_groups = {
    "A": ["United States", "Panama", "Algeria"],
    "B": ["Mexico", "Jamaica", "Venezuela"],
    "C": ["Canada", "Honduras", "Uzbekistan"],
    "D": ["Brazil", "Ecuador", "Bahrain"],
    "E": ["Argentina", "Chile", "Australia"],
    "F": ["Colombia", "Paraguay", "Saudi Arabia"],
    "G": ["Uruguay", "Bolivia", "Japan"],
    "H": ["France", "Belgium", "Morocco"],
    "I": ["Spain", "Portugal", "New Zealand"],
    "J": ["England", "Netherlands", "Senegal"],
    "K": ["Germany", "Austria", "Cameroon"],
    "L": ["Italy", "Croatia", "Egypt"],
    "M": ["Serbia", "Slovenia", "South Korea"],
    "N": ["Poland", "Czechia", "Nigeria"],
    "O": ["Turkey", "Hungary", "Tunisia"],
    "P": ["Iran", "Ivory Coast", "Inter-confederation Play-off Winner"],
}

# Flatten into a DataFrame
rows = []
for group, teams in wc2026_groups.items():
    for team in teams:
        rows.append({"group": group, "team": team})

wc2026 = pd.DataFrame(rows)
print("✅ WC 2026 structure built")
print(wc2026)

✅ WC 2026 structure built
   group                                 team
0      A                        United States
1      A                               Panama
2      A                              Algeria
3      B                               Mexico
4      B                              Jamaica
5      B                            Venezuela
6      C                               Canada
7      C                             Honduras
8      C                           Uzbekistan
9      D                               Brazil
10     D                              Ecuador
11     D                              Bahrain
12     E                            Argentina
13     E                                Chile
14     E                            Australia
15     F                             Colombia
16     F                             Paraguay
17     F                         Saudi Arabia
18     G                              Uruguay
19     G                              Bolivia
20     G

In [33]:
print("=== WC 2026 STRUCTURE ===")
print("Shape            :", wc2026.shape)
print("Unique teams     :", wc2026["team"].nunique())
print("Unique groups    :", wc2026["group"].nunique())
print("\nTeams per group:")
print(wc2026.groupby("group")["team"].apply(list))

=== WC 2026 STRUCTURE ===
Shape            : (48, 2)
Unique teams     : 48
Unique groups    : 16

Teams per group:
group
A                     [United States, Panama, Algeria]
B                         [Mexico, Jamaica, Venezuela]
C                       [Canada, Honduras, Uzbekistan]
D                           [Brazil, Ecuador, Bahrain]
E                        [Argentina, Chile, Australia]
F                   [Colombia, Paraguay, Saudi Arabia]
G                            [Uruguay, Bolivia, Japan]
H                           [France, Belgium, Morocco]
I                       [Spain, Portugal, New Zealand]
J                      [England, Netherlands, Senegal]
K                         [Germany, Austria, Cameroon]
L                              [Italy, Croatia, Egypt]
M                      [Serbia, Slovenia, South Korea]
N                           [Poland, Czechia, Nigeria]
O                           [Turkey, Hungary, Tunisia]
P    [Iran, Ivory Coast, Inter-confederation Play-o...

In [34]:
wc2026.to_csv(CLEAN_DIR + "wc2026_structure.csv", index=False)
print("✅ Saved → data/clean/wc2026_structure.csv")

✅ Saved → data/clean/wc2026_structure.csv


In [35]:
# Get all unique team names from matches (combined home and away)
match_teams = set(matches["home_team"].unique()) | set(matches["away_team"].unique())

# Get all WC 2026 team names
wc_teams = set(wc2026["team"].unique())

# Find WC teams that do NOT appear in match data
missing_from_matches = wc_teams - match_teams

print(f"WC teams not found in match data ({len(missing_from_matches)}):")
for t in sorted(missing_from_matches):
    print(" ✗", t)

WC teams not found in match data (2):
 ✗ Czechia
 ✗ Inter-confederation Play-off Winner


In [36]:
# Get all unique team names from ranking
ranking_teams = set(ranking["team"].unique())

# Find WC teams that do NOT appear in ranking data
missing_from_ranking = wc_teams - ranking_teams

print(f"WC teams not found in ranking data ({len(missing_from_ranking)}):")
for t in sorted(missing_from_ranking):
    print(" ✗", t)

WC teams not found in ranking data (5):
 ✗ Inter-confederation Play-off Winner
 ✗ Iran
 ✗ Ivory Coast
 ✗ South Korea
 ✗ United States


In [41]:
# Search for the approximate name used in ranking data for each mismatch
search_terms = ["iran", "korea", "ivory", "cote", "united states", "america", "czech"]

ranking_teams_list = sorted(ranking["team"].unique())

for term in search_terms:
    matches_found = [t for t in ranking_teams_list if term.lower() in t.lower()]
    print(f"'{term}' → {matches_found}")

'iran' → ['Iran']
'korea' → ['Korea Dpr', 'South Korea']
'ivory' → ['Ivory Coast']
'cote' → []
'united states' → ['United States']
'america' → ['American Samoa']
'czech' → ['Czech Republic']


In [42]:
# Search for the approximate name used in ranking data for each mismatch
search_terms = ["iran", "korea", "ivory", "cote", "united states", "america", "czech"]

ranking_teams_list = sorted(ranking["team"].unique())

for term in search_terms:
    matches_found = [t for t in ranking_teams_list if term.lower() in t.lower()]
    print(f"'{term}' → {matches_found}")

'iran' → ['Iran']
'korea' → ['Korea Dpr', 'South Korea']
'ivory' → ['Ivory Coast']
'cote' → []
'united states' → ['United States']
'america' → ['American Samoa']
'czech' → ['Czech Republic']


In [43]:
# Minimal mapping — only what's actually needed based on our investigation
name_map = {
    "Czech Republic" : "Czechia",   # match & ranking → align to WC2026 name
}

# Apply to matches
matches["home_team"] = matches["home_team"].replace(name_map)
matches["away_team"] = matches["away_team"].replace(name_map)

# Apply to ranking
ranking["team"] = ranking["team"].replace(name_map)

# wc2026 already uses "Czechia" — no change needed there

print("✅ Name mapping applied")

✅ Name mapping applied


In [45]:
# Check what wc2026 actually has for Czechia/Czech Republic
print("WC2026 team list (sorted):")
print(sorted(wc2026["team"].unique()))

# Check what matches has
print("\nDoes 'Czechia' appear in matches?", 
      "Czechia" in set(matches["home_team"]) | set(matches["away_team"]))
print("Does 'Czech Republic' appear in matches?", 
      "Czech Republic" in set(matches["home_team"]) | set(matches["away_team"]))

# Check ranking
print("\nDoes 'Czechia' appear in ranking?", "Czechia" in set(ranking["team"]))
print("Does 'Czech Republic' appear in ranking?", "Czech Republic" in set(ranking["team"]))

WC2026 team list (sorted):
['Algeria', 'Argentina', 'Australia', 'Austria', 'Bahrain', 'Belgium', 'Bolivia', 'Brazil', 'Cameroon', 'Canada', 'Chile', 'Colombia', 'Croatia', 'Czech Republic', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Honduras', 'Hungary', 'Inter-confederation Play-off Winner', 'Iran', 'Italy', 'Ivory Coast', 'Jamaica', 'Japan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Nigeria', 'Panama', 'Paraguay', 'Poland', 'Portugal', 'Saudi Arabia', 'Senegal', 'Serbia', 'Slovenia', 'South Korea', 'Spain', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan', 'Venezuela']

Does 'Czechia' appear in matches? True
Does 'Czech Republic' appear in matches? False

Does 'Czechia' appear in ranking? True
Does 'Czech Republic' appear in ranking? False


In [46]:
# wc2026 was never updated — fix it directly
wc2026["team"] = wc2026["team"].replace({"Czech Republic": "Czechia"})

# Re-check
wc_teams      = set(wc2026["team"].unique())
match_teams   = set(matches["home_team"].unique()) | set(matches["away_team"].unique())
ranking_teams = set(ranking["team"].unique())

missing_from_matches = wc_teams - match_teams
missing_from_ranking = wc_teams - ranking_teams

print(f"Still missing from matches ({len(missing_from_matches)}): {sorted(missing_from_matches)}")
print(f"Still missing from ranking ({len(missing_from_ranking)}): {sorted(missing_from_ranking)}")

Still missing from matches (1): ['Inter-confederation Play-off Winner']
Still missing from ranking (1): ['Inter-confederation Play-off Winner']


In [47]:
matches.to_csv(CLEAN_DIR + "matches_clean.csv", index=False)
ranking.to_csv(CLEAN_DIR + "ranking_clean.csv", index=False)
wc2026.to_csv(CLEAN_DIR + "wc2026_structure.csv", index=False)

print("✅ All 3 clean files re-saved with standardized team names")

✅ All 3 clean files re-saved with standardized team names
